In [ ]:
# default_exp paper.cnn.overfitting

# 4.5. Overfitting in action (CNN)

> CNN model performance trained on all Soil Taxonomy Orders and looking at overfitting in action when trained over 200 epochs

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Python utils
import math
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y
from mirzai.training.cnn import (Model, weights_init)
from mirzai.data.torch import (DataLoaders, SNV_transform, SpectralDataset)
from mirzai.training.cnn import Learner

from fastcore.transform import compose

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split

# Deep Learning stack
import torch
from torch.nn import MSELoss
from torch.optim import Adam
from torch.optim.lr_scheduler import CyclicLR

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
# Is a GPU available?
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
print(f'Runtime is: {device}')

Runtime is: cpu


In [ ]:
seeds = range(20)
criterion = MSELoss() # Mean Squared Error loss
min_epoch, max_epoch = 10, 210
base_lr, max_lr = 3e-5, 1e-3

epoch_inc = 10
step_size_up = 5

if device.type == 'cpu':
    min_epoch, max_epoch = 1, 3
    epoch_inc = 1
    seeds = range(2)
    
tax_of_interest = tax_lookup.copy()
tax_of_interest['all'] = tax_of_interest.pop('oxisols') # Too few Oxisols for now in the KSSL db    

mses_by_order = np.zeros((len(seeds),
                          len(range(min_epoch, max_epoch, epoch_inc)),
                          len(tax_of_interest),
                          2))

### Run

In [ ]:
for seed in tqdm(seeds):
    print("************")
    print('Seed: ', seed)
    
    # 1. Initialize a "fresh" new CNN
    model = Model(X.shape[1], out_channel=16).to(device)
    opt = Adam(model.parameters(), lr=1e-4)
    model = model.apply(weights_init)
    scheduler = CyclicLR(opt, base_lr=base_lr, max_lr=max_lr,
                         step_size_up=step_size_up, mode='triangular',
                         cycle_momentum=False)

    # 2. Starting from min_epoch, train additional 10 epochs then stop when reaching max_epoch 
    for epoch_idx, epoch in enumerate(range(min_epoch, max_epoch, epoch_inc)):
        print('Epoch: ', epoch)
        data = train_test_split(X, y, depth_order, test_size=0.1, random_state=seed)
        X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = data
        dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
        training_generator, test_generator = dls.loaders()
            
        # Train addition `epoch_inc` epochs
        learner = Learner(model, criterion, opt, n_epochs=epoch_inc, 
                          scheduler=scheduler, verbose=True)
        model, losses = learner.fit(training_generator, test_generator) 
               
        # 3. Evaluate on train & test sets on each Soil Taxonomy orders of interest
        for tax_idx in range(len(tax_of_interest)):
            if tax_idx == 9:
                mask_train = np.ones(len(depth_order_train), dtype=bool)
                mask_test = np.ones(len(depth_order_test), dtype=bool)
            else:
                mask_train = depth_order_train[:, 1] == tax_idx
                mask_test = depth_order_test[:, 1] == tax_idx
            if np.sum(mask_test) != 0:
                training_set_order = SpectralDataset(X_train[mask_train, :], y_train[mask_train], transform=SNV_transform())
                test_set_order = SpectralDataset(X_test[mask_test, :], y_test[mask_test], transform=SNV_transform())

                #y_hat_train, y_true_train = predict(model, training_set_order)
                y_hat_train, y_true_train = learner.predict(training_set_order)
                mse_train = eval_reg(y_true_train, y_hat_train, is_log=True)['mse']

                #y_hat_test, y_true_test = predict(model, test_set_order)
                y_hat_test, y_true_test = learner.predict(test_set_order)
                mse_test = eval_reg(y_true_test, y_hat_test, is_log=True)['mse']

                mses_by_order[seed, epoch_idx, tax_idx, 0] = mse_train
                mses_by_order[seed, epoch_idx, tax_idx, 1] = mse_test
            else:
                deltas_mse[seed, epoch_idx, tax_idx] = np.nan


  0%|          | 0/2 [00:00<?, ?it/s]

************
Seed:  0
Epoch:  1
Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.125955143588718
 Validation loss: 0.10428656276965899


TypeError: 'int' object is not callable

## Save

In [ ]:
MONITORING_PATH = Path('nameofyourfolder')
with open(MONITORING_PATH/'mses_by_order_90_10.pickle'), 'wb') as f: 
    pickle.dump(mses_by_order, f)